# AI Resume Screening System (LangChain + LangSmith)

This notebook builds an end-to-end pipeline:
Resume → Extract → Match → Score → Explain

In [ ]:
# Install dependencies (run once)
!pip install langchain langchain-openai langsmith python-dotenv

In [ ]:
# Imports
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os

# Set your API key here
os.environ['OPENAI_API_KEY'] = 'your_key_here'

## Sample Data

In [ ]:
job_description = '''
Data Scientist Role
Requirements:
- Python
- Machine Learning
- Pandas
- SQL
'''

strong_resume = '''
Experienced Data Scientist with skills in Python, Machine Learning, Pandas, SQL.
Worked on ML projects.
'''

average_resume = '''
Knows Python and Pandas.
Basic ML knowledge.
'''

weak_resume = '''
Good communication skills.
Basic computer knowledge.
'''

## Step 1: Skill Extraction

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini')

extract_prompt = PromptTemplate.from_template('''
Extract skills, tools, experience from resume.
Do not assume anything.

Resume:
{resume}
''')

def extract(resume):
    return (extract_prompt | llm).invoke({'resume': resume})

extract(strong_resume)

## Step 2: Matching

In [ ]:
match_prompt = PromptTemplate.from_template('''
Compare resume and job description.
Return matching and missing skills.

JD:
{jd}

Resume:
{resume}
''')

def match(resume, jd):
    return (match_prompt | llm).invoke({'resume': resume, 'jd': jd})

match(strong_resume, job_description)

## Step 3: Scoring

In [ ]:
score_prompt = PromptTemplate.from_template('''
Give a score from 0-100 based on match.

Data:
{data}
''')

def score(data):
    return (score_prompt | llm).invoke({'data': data})

m = match(strong_resume, job_description)
score(m)

## Step 4: Explanation

In [ ]:
explain_prompt = PromptTemplate.from_template('''
Explain the score.

Score:
{score}

Data:
{data}
''')

def explain(score_val, data):
    return (explain_prompt | llm).invoke({'score': score_val, 'data': data})

s = score(m)
explain(s, m)

## Final Pipeline

In [ ]:
def pipeline(resume, jd):
    e = extract(resume)
    m = match(resume, jd)
    s = score(m)
    ex = explain(s, m)
    return {'score': s, 'explanation': ex}

pipeline(strong_resume, job_description)